# Notebook 06 — Multi-agent systems: a small team of AI agents

**Time:** about 40 minutes

**What you will learn**
- What an "agent" is in practice: **the same model with a different role** (system message)
- How agents pass work to each other: researcher → writer → reviewer
- How to stop a team from looping forever
- Why documents from outside must be treated with care (*prompt injection*)

### The big idea
Like a real team, each agent has **one clear job**:

| Agent | Job | Gets | Gives |
|---|---|---|---|
| 🔎 Researcher | Pick out the facts | Documents | Notes with sources like [DOC-13] |
| ✍️ Writer | Write a short report | The notes | A report |
| 🧐 Reviewer | Check the report | Notes + report | "approved" or feedback |

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [1]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [2]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 1 — An agent is just a role + the model

In [3]:
RESEARCHER = ("You are a careful researcher. Write 4 short "
              "bullet-point facts. After each fact put its "
              "source, like [DOC-13]. Use only the documents.")
WRITER = ("You are a clear writer. Write a report of about "
          "120 words from the notes. Keep the [DOC-xx] sources.")
REVIEWER = ("You are a strict reviewer. Check the report uses "
            "only facts from the notes and keeps their sources.")

message_log = []          # who sent what to whom

def send(sender, receiver, text):
    message_log.append({"from": sender, "to": receiver,
                        "text": text[:80] + "..."})

## Part 2 — The shared documents, and a safety check

The documents come from `research_documents.json`. **One of them is a trap:** it contains a hidden instruction
("ignore all previous instructions…"). Text from outside is **data, not instructions** — so we filter it out.

In [4]:
import json

path = "../datasets/research_documents.json"
documents = json.load(open(path, encoding="utf-8"))["documents"]
farm_docs = [d for d in documents if d["topic"] == "agriculture"]

WARNING_SIGNS = ["ignore all previous instructions",
                 "ignore previous instructions",
                 "api key", "cite this"]

def looks_suspicious(text):
    text = text.lower()
    for sign in WARNING_SIGNS:
        if sign in text:
            return True
    return False

In [5]:
safe_docs = []
for doc in farm_docs:
    if looks_suspicious(doc["text"]):
        print("🚫 Skipped (suspicious):", doc["doc_id"], doc["title"])
    else:
        safe_docs.append(doc)
        print("✅ Using:", doc["doc_id"], doc["title"])

evidence = ""
for d in safe_docs:
    evidence += f"[{d['doc_id']}] {d['title']}: {d['text']}\n\n"

✅ Using: DOC-13 Crop monitoring with imagery
✅ Using: DOC-14 Advisory assistants for farmers
✅ Using: DOC-15 Precision irrigation and resource use
✅ Using: DOC-16 Supply chain and market information
🚫 Skipped (suspicious): DOC-17 Farm forum post (unverified)


💡 A keyword list is only a **first** defence. It can miss clever tricks, and it can block innocent text:
if we added the word *"reveal"* to the list, DOC-13 ("imagery can reveal…") would be wrongly removed. Real systems also
tell the model that documents are data, and check the final output with code (Part 5).

## Part 3 — Researcher → Writer

In [6]:
TOPIC = "How can AI help small farmers?"

notes = ask(f"Topic: {TOPIC}\n\nDocuments:\n{evidence}",
            system=RESEARCHER)
send("researcher", "writer", notes)
print("🔎 RESEARCHER NOTES:\n", notes)

🔎 RESEARCHER NOTES:
 Here are four short bullet-point facts on how AI can help small farmers, along with their sources:

• AI-powered crop monitoring with imagery can help small farmers identify crop stress, nutrient deficiency, and early signs of disease, enabling them to take timely action to prevent damage. [DOC-13]
• Advisory assistants powered by AI can provide small farmers with accurate and localized advice on planting dates, pest management, and fertiliser use, helping them make informed decisions. [DOC-14]
• AI-driven precision irrigation and resource use can help small farmers optimize water usage, reducing waste and increasing yields, particularly in areas where water is scarce. [DOC-15]
• AI-powered supply chain and market information can help small farmers access better prices for their produce, plan transport more efficiently, and reduce post-harvest losses, improving their overall livelihoods. [DOC-16]


In [7]:
report = ask(f"Topic: {TOPIC}\n\nNotes:\n{notes}", system=WRITER)
send("writer", "reviewer", report)
print("✍️ REPORT:\n", report)

✍️ REPORT:
 **Empowering Small Farmers through AI: A Game-Changer for Sustainable Agriculture**

Artificial intelligence (AI) is poised to revolutionize the lives of small farmers worldwide. By leveraging AI-powered technologies, these farmers can gain a competitive edge in the agricultural sector. Key benefits of AI for small farmers include:

* Early crop monitoring and disease detection, enabling timely interventions to prevent damage
* Personalized advisory services on planting, pest management, and fertiliser use
* Precision irrigation and resource optimization, reducing waste and increasing yields
* Access to better market prices, efficient transport planning, and reduced post-harvest losses

By harnessing the power of AI, small farmers can improve their livelihoods, increase productivity, and contribute to sustainable agriculture practices.


## Part 4 — Reviewer → Writer, with a limit

The reviewer answers in JSON: `{"approved": true/false, "feedback": "..."}`.
If it is not approved, the writer revises. **At most 2 revisions**, so the team cannot loop forever.

In [8]:
from pydantic import BaseModel

class Review(BaseModel):
    approved: bool
    feedback: str

def review(report):
    content = f"NOTES:\n{notes}\n\nREPORT:\n{report}"
    messages = [{"role": "system", "content": REVIEWER},
                {"role": "user", "content": content}]
    reply = ollama.chat(model=MODEL, messages=messages,
                        format=Review.model_json_schema(),
                        options={"temperature": 0})
    try:
        return Review.model_validate_json(reply.message.content)
    except Exception:
        return Review(approved=False,
                      feedback="The review could not be read.")

In [ ]:
for round_number in range(1, 3):          # at most 2 revisions
    verdict = review(report)
    send("reviewer", "writer", verdict.feedback)
    print(f"🧐 Review {round_number}: approved={verdict.approved}")
    print("   Feedback:", verdict.feedback)
    if verdict.approved:
        break
    report = ask(f"Revise the report using this feedback.\n\n"
                 f"FEEDBACK: {verdict.feedback}\n\n"
                 f"NOTES:\n{notes}\n\nREPORT:\n{report}",
                 system=WRITER)
    send("writer", "reviewer", report)

print("\nFINAL REPORT:\n", report)

## Part 5 — Check the sources with code
Every `[DOC-xx]` in the report must be one of the documents we actually gave the team.

In [ ]:
import re

cited = set(re.findall(r"DOC-\d+", report))
allowed = {d["doc_id"] for d in safe_docs}

print("Sources cited  :", sorted(cited))
print("Allowed        :", sorted(allowed))
print("Made-up sources:", sorted(cited - allowed) or "none ✅")
print("Trap document cited?", "DOC-17" in cited)

## Part 6 — How the team talked

In [ ]:
import pandas as pd
pd.DataFrame(message_log)

## 💬 Discuss
- The team made more model calls than one agent would. Was the result better?
- What would happen without the "at most 2 revisions" limit?
- Why did we remove the suspicious document **before** any agent saw it?

### Team shapes (from the course)
| Pattern | How it works |
|---|---|
| Line (pipeline) | A → B → C, like this notebook |
| Manager and workers | A manager hands tasks to specialists and collects results |
| Everyone talks to everyone | Flexible but hard to predict — rarely a good first choice |

## ✍️ Try it yourself
1. Change `TOPIC` and the topic filter to `"student_services"`.
2. Make the reviewer stricter: *"Reject the report if it has fewer than 3 different sources."*

## ✅ Summary
- An agent = the model + a role. A team = agents passing work along.
- Limits stop endless loops; code checks catch made-up sources.
- Outside text is data: filter or mark it, never obey it.

**Next:** Notebook 07 — the **capstone**: a complete research assistant.